# Customer Segmentation — Evaluation, Visualisation & Insights
## Data Mining & Machine Learning | February 2026

**Sprint:** 3 | **User Stories:** US-5.1, US-5.2, US-5.3, US-5.4

---

### Notebook Objectives
1. Compare all models on Silhouette, Davies-Bouldin, Calinski-Harabasz (US-5.1)
2. PCA 2D scatter plots for all three algorithms (US-5.3)
3. Cluster profiling and persona assignment (US-5.4)
4. Post-hoc validation using Response column
5. Select and save the best model as `best_model.pkl`

## Section 1 — Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from src.profiling import profile_clusters, assign_persona
from src.evaluation import evaluate_clustering

sns.set_theme(style='whitegrid')
PROCESSED_DIR = '../data/processed'
MODELS_DIR    = '../models'
FIGURES_DIR   = '../figures'

# Load data
X       = np.load(f'{PROCESSED_DIR}/X_scaled.npy')
X_2d    = np.load(f'{PROCESSED_DIR}/X_pca2d.npy')
df_eng  = pd.read_csv(f'{PROCESSED_DIR}/df_cleaned.csv')

# Load labels
km_labels   = np.load(f'{MODELS_DIR}/kmeans_labels.npy')
hier_labels = np.load(f'{MODELS_DIR}/hierarchical_labels.npy')
db_labels   = np.load(f'{MODELS_DIR}/dbscan_labels.npy')
baseline    = joblib.load(f'{MODELS_DIR}/baseline_results.pkl')

print('All files loaded successfully')

## Section 2 — Metric Comparison Table (US-5.1, US-5.2)

In [ ]:
# Recompute metrics for all models
km_metrics   = evaluate_clustering(X, km_labels)
hier_metrics = evaluate_clustering(X, hier_labels)
db_metrics   = evaluate_clustering(X, db_labels, ignore_noise=True)
base_metrics = baseline['metrics']

comparison = pd.DataFrame({
    'Model':             ['Baseline KMeans(k=2)', 'K-Means (optimal k)',
                          'Hierarchical (Ward)',   'DBSCAN'],
    'n_clusters':        [base_metrics['n_clusters'], km_metrics['n_clusters'],
                          hier_metrics['n_clusters'], db_metrics['n_clusters']],
    'Silhouette ↑':      [base_metrics['silhouette'], km_metrics['silhouette'],
                          hier_metrics['silhouette'], db_metrics['silhouette']],
    'Davies-Bouldin ↓':  [base_metrics['davies_bouldin'], km_metrics['davies_bouldin'],
                          hier_metrics['davies_bouldin'], db_metrics['davies_bouldin']],
    'Calinski-Harabasz ↑': [base_metrics['calinski_harabasz'], km_metrics['calinski_harabasz'],
                             hier_metrics['calinski_harabasz'], db_metrics['calinski_harabasz']],
}).round(4)

print('=== Model Comparison Table ===')
print(comparison.to_string(index=False))

In [ ]:
# Identify winner
best_sil_idx = comparison['Silhouette ↑'].idxmax()
best_model_name = comparison.loc[best_sil_idx, 'Model']
print(f'\nBest model by Silhouette Score: {best_model_name}')
print(f'Silhouette: {comparison.loc[best_sil_idx, "Silhouette ↑"]:.4f}')

# Map to labels
label_map = {
    'Baseline KMeans(k=2)':  baseline['labels'],
    'K-Means (optimal k)':   km_labels,
    'Hierarchical (Ward)':   hier_labels,
    'DBSCAN':                db_labels,
}
best_labels = label_map[best_model_name]

## Section 3 — PCA 2D Scatter Plots (US-5.3)

In [ ]:
# 2D PCA scatter for all three algorithms + baseline
all_models = [
    ('Baseline KMeans(k=2)', baseline['labels']),
    ('K-Means (optimal k)',  km_labels),
    ('Hierarchical (Ward)',  hier_labels),
    ('DBSCAN',               db_labels),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
palette = sns.color_palette('tab10', 10)

for ax, (name, labels) in zip(axes, all_models):
    unique_labels = sorted(set(labels))
    for lbl in unique_labels:
        mask = labels == lbl
        color = 'lightgrey' if lbl == -1 else palette[lbl % 10]
        marker = 'x' if lbl == -1 else 'o'
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   c=[color], marker=marker, s=15, alpha=0.6,
                   label=f'Noise' if lbl == -1 else f'Cluster {lbl}')
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.legend(fontsize=7, markerscale=1.5)

plt.suptitle('PCA 2D Cluster Visualisation — All Models', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/pca_cluster_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 4 — Cluster Profiling & Persona Assignment (US-5.4)

In [ ]:
# Profile clusters using best model labels
profile = profile_clusters(df_eng, best_labels)
print(f'Cluster profile for: {best_model_name}')
print()
print(profile[['Cluster', 'Persona', 'Count', 'Avg_Age',
               'Median_Income', 'TotalSpent', 'Recency',
               'DealRate', 'WebShare', 'StoreShare']].round(2).to_string(index=False))

In [ ]:
# Bar chart — TotalSpent per cluster coloured by persona
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_to_plot = [
    ('TotalSpent',     'Avg TotalSpent by Cluster'),
    ('Median_Income',  'Median Income by Cluster'),
    ('Recency',        'Avg Recency by Cluster'),
]

colors = sns.color_palette('Set2', len(profile))

for ax, (col, title) in zip(axes, metrics_to_plot):
    bars = ax.bar(profile['Cluster'].astype(str) + '\n' + profile['Persona'],
                  profile[col], color=colors, edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('Cluster — Persona')
    ax.set_ylabel(col)
    for bar, val in zip(bars, profile[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{val:.0f}', ha='center', va='bottom', fontsize=8)

plt.suptitle(f'Cluster Profiles — {best_model_name}', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/cluster_profiles.png', dpi=150)
plt.show()

In [ ]:
# Channel share radar-style grouped bar chart
channel_cols = ['WebShare', 'CatalogShare', 'StoreShare']
channel_data = profile[['Cluster', 'Persona'] + channel_cols].copy()

x = np.arange(len(channel_cols))
width = 0.8 / len(profile)

fig, ax = plt.subplots(figsize=(12, 5))
for i, row in profile.iterrows():
    vals = [row['WebShare'], row['CatalogShare'], row['StoreShare']]
    ax.bar(x + i * width, vals, width, label=f"Cluster {int(row['Cluster'])} — {row['Persona']}",
           color=colors[i % len(colors)])

ax.set_xticks(x + width * len(profile) / 2)
ax.set_xticklabels(['Web Share', 'Catalog Share', 'Store Share'])
ax.set_title('Channel Preference by Cluster')
ax.set_ylabel('Share (0–1)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/channel_shares.png', dpi=150)
plt.show()

## Section 5 — Post-Hoc Validation with Response

In [ ]:
# Validate segments using Response (campaign acceptance)
# This is the only place Response is used — never as a clustering input
df_val = df_eng.copy()
df_val['Cluster'] = best_labels

# Load original Response column (may not be in df_cleaned — load from raw)
df_raw = pd.read_csv('../data/raw/marketing_campaign.csv',
                     sep='\t', encoding='latin-1')
df_raw = df_raw[~df_raw['Marital_Status'].isin(['Absurd','YOLO','Alone'])]
df_raw = df_raw[df_raw['Income'] < 600_000]

df_val['Response'] = df_raw['Response'].values

response_by_cluster = df_val.groupby('Cluster')['Response'].agg(['mean','sum','count'])
response_by_cluster.columns = ['Acceptance_Rate', 'Accepted', 'Total']
response_by_cluster = response_by_cluster.merge(
    profile[['Cluster','Persona']], on='Cluster')
response_by_cluster['Acceptance_Rate'] = response_by_cluster['Acceptance_Rate'].map('{:.2%}'.format)
print('Campaign Acceptance Rate by Cluster (post-hoc validation):')
print(response_by_cluster.to_string(index=False))

## Section 6 — Save Best Model

In [ ]:
# Determine which model file to copy as best_model.pkl
model_file_map = {
    'Baseline KMeans(k=2)':  'baseline_results.pkl',
    'K-Means (optimal k)':   'kmeans.pkl',
    'Hierarchical (Ward)':   'hierarchical.pkl',
    'DBSCAN':                'dbscan.pkl',
}

best_model_file = model_file_map[best_model_name]
best_model_obj  = joblib.load(f'{MODELS_DIR}/{best_model_file}')
joblib.dump(best_model_obj, f'{MODELS_DIR}/best_model.pkl')
np.save(f'{MODELS_DIR}/best_labels.npy', best_labels)
profile.to_csv(f'{MODELS_DIR}/cluster_profiles.csv', index=False)

print(f'Best model  : {best_model_name}')
print(f'Saved as    : models/best_model.pkl')
print(f'Labels saved: models/best_labels.npy')
print(f'Profiles    : models/cluster_profiles.csv')

## Section 7 — Final Summary

| Item | Value |
|---|---|
| Best Model | *(auto-selected above)* |
| n_clusters | *(from profile table)* |
| Silhouette Score | *(from comparison table)* |
| Figures saved | `pca_cluster_comparison.png`, `cluster_profiles.png`, `channel_shares.png` |
| Model saved | `models/best_model.pkl` |

> **Next:** Proceed to Sprint 4 — build `app.py` Streamlit dashboard using
> `models/best_model.pkl`, `models/pipeline.pkl`, and `models/cluster_profiles.csv`.